# Experiment 08 — Task 5: TabICL with Improved Features

Task 3 ran conventional models (LinearSVM, RF, GB) on the full feature set. Task 5 takes the best-performing model overall — TabICL — and runs it on the improved feature set informed by the feature importance analysis in exp_07.

Two things change from the Task 3 TabICL run (exp_06):
1. **Featurizer** — `"improved"` instead of `"full"`: proper admission source grouping, collapsed rare race categories
2. **Feature selection** — top features by permutation importance from exp_07, removing noise and artefacts

Same target (any readmission), same 5-fold CV + test set setup, same random seed. Results go into the final comparison table.

In [ ]:
import sys, os
if 'google.colab' in str(get_ipython()):
    REPO = 'diabetes-uci-dataset'
    REPO_URL = 'https://github.com/byambaa0325/diabetes-uci-dataset.git'
    if not os.path.exists(REPO):
        os.system(f'git clone ${REPO_URL}')
    os.chdir(REPO)
    os.system('pip install -q -r requirements.txt')
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
    if root not in sys.path:
        sys.path.insert(0, root)

In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path

from src.pipeline import cross_validate, run_pipeline
from src.models.registry import MODEL_REGISTRY

sns.set_theme(style='whitegrid')

assert 'tabicl' in MODEL_REGISTRY, 'tabicl not in registry — pip install tabicl'

TEST_RATIO  = 0.15
CV_SPLITS   = 5
RANDOM_SEED = 42
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

CV_CHECKPOINT_DIR = Path('../../outputs/checkpoints/task5_tabicl_improved_cv')
TEST_CHECKPOINT   = Path('../../outputs/checkpoints/task5_tabicl_improved_test.json')
CV_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU


---
## Feature selection from exp_07

Load permutation importance rankings and take the top N. If exp_07 hasn't been run yet, falls back to a clinically motivated list.

In [3]:
PERM_PATH  = Path('../../outputs/tables/permutation_importance.csv')
TOP_N      = 30

if PERM_PATH.exists():
    perm_df          = pd.read_csv(PERM_PATH, index_col=0)
    mean_imp         = perm_df.mean(axis=1).sort_values(ascending=False)
    selected_features = mean_imp.head(TOP_N).index.tolist()
    print(f'Loaded permutation importance — top {TOP_N} features selected')
else:
    print('permutation_importance.csv not found — using clinical fallback list')
    print('Run exp_07 first for a data-driven selection')
    selected_features = [
        'number_inpatient', 'prior_hospital_load', 'number_emergency', 'number_outpatient',
        'num_diagnoses', 'num_medications', 'total_meds_prescribed', 'time_in_hospital',
        'num_lab_procedures', 'num_procedures', 'lab_to_procedure_ratio',
        'age', 'age_x_medications',
        'a1cresult', 'a1c_poor_control', 'max_glu_serum', 'insulin', 'insulin_adjusted',
        'discharge_group_home', 'discharge_group_transfer', 'discharge_group_expired',
        'admission_group_emergency', 'admission_group_other',
        'admission_source_group_emergency', 'admission_source_group_transfer',
        'change', 'diabetesmed',
        'diag_1_group_Circulatory', 'diag_1_group_Diabetes', 'diag_1_group_Respiratory',
    ]

print(f'\n{len(selected_features)} features selected:')
print(selected_features)

permutation_importance.csv not found — using clinical fallback list
Run exp_07 first for a data-driven selection

30 features selected:
['number_inpatient', 'prior_hospital_load', 'number_emergency', 'number_outpatient', 'num_diagnoses', 'num_medications', 'total_meds_prescribed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'lab_to_procedure_ratio', 'age', 'age_x_medications', 'a1cresult', 'a1c_poor_control', 'max_glu_serum', 'insulin', 'insulin_adjusted', 'discharge_group_home', 'discharge_group_transfer', 'discharge_group_expired', 'admission_group_emergency', 'admission_group_other', 'admission_source_group_emergency', 'admission_source_group_transfer', 'change', 'diabetesmed', 'diag_1_group_Circulatory', 'diag_1_group_Diabetes', 'diag_1_group_Respiratory']


In [4]:
BASE_CONFIG = {
    'name':               'task5_tabicl_improved',
    'model':              'tabicl',
    'wandb_project':      'applied-ai-coursework',
    'featurizer':         'improved',
    'selected_features':   selected_features,
    'test_ratio':          TEST_RATIO,
    'predict_batch_size':  256,
    'model_params': {
        'batch_size':    2,
        'n_estimators':  8,
        'device':        DEVICE,
        'use_amp':      'auto',
        'offload_mode': 'cpu',
        'kv_cache':      False,
        'random_state':  RANDOM_SEED,
    },
}

print('Featurizer      :', BASE_CONFIG['featurizer'])
print('Features        :', len(selected_features))
print('TabICL device   :', DEVICE)

Featurizer      : improved
Features        : 30
TabICL device   : cuda


---
## 5-fold CV — resumable

In [5]:
summary_path = CV_CHECKPOINT_DIR / 'summary.json'

if summary_path.exists():
    print('CV already complete — loading from checkpoint.')
    cv_result = json.loads(summary_path.read_text())
else:
    completed = sorted(CV_CHECKPOINT_DIR.glob('fold_*.json'))
    if completed:
        print(f'Resuming CV — {len(completed)}/{CV_SPLITS} folds already done.')
    else:
        print(f'Starting {CV_SPLITS}-fold CV...')

    cv_result = cross_validate(
        BASE_CONFIG,
        n_splits=CV_SPLITS,
        random_state=RANDOM_SEED,
        checkpoint_dir=CV_CHECKPOINT_DIR,
    )
    summary_path.write_text(json.dumps(cv_result['summary'], indent=2))
    print('CV complete.')

s = cv_result['summary']
print(f"\nCV Results ({CV_SPLITS} folds):")
print(f"  ROC-AUC    : {s['roc_auc_mean']:.4f} ± {s['roc_auc_std']:.4f}")
print(f"  F1         : {s['f1_mean']:.4f} ± {s['f1_std']:.4f}")
print(f"  Recall(pos): {s['recall_pos_mean']:.4f} ± {s['recall_pos_std']:.4f}")
print(f"  Precision  : {s['precision_mean']:.4f} ± {s['precision_std']:.4f}")

Resuming CV — 2/5 folds already done.
  fold 1/5  [resumed from checkpoint]
  fold 2/5  [resumed from checkpoint]
  fold 3/5  

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\byamb\_netrc.
wandb: Currently logged in as: byambaa-bayarmandakh-25 (byambaa-bayarmandakh-25-ucl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


train_log_loss,▁
val_log_loss,▁
train_log_loss,0.57603
val_log_loss,0.6328


Run saved: E:\UCL-Workspaces\applied-ai-coursework\runs\2026-03-09_23-57-51_e6607f_task5_tabicl_improved
  [saved]
  fold 4/5  

train_log_loss,▁
val_log_loss,▁
train_log_loss,0.5704
val_log_loss,0.63177


Run saved: E:\UCL-Workspaces\applied-ai-coursework\runs\2026-03-10_00-44-49_622855_task5_tabicl_improved
  [saved]
  fold 5/5  

train_log_loss,▁
val_log_loss,▁
train_log_loss,0.57389
val_log_loss,0.63132


Run saved: E:\UCL-Workspaces\applied-ai-coursework\runs\2026-03-10_01-31-49_8605aa_task5_tabicl_improved
  [saved]
CV complete.

CV Results (5 folds):
  ROC-AUC    : 0.6683 ± 0.0028
  F1         : 0.6215 ± 0.0011
  Recall(pos): 0.8918 ± 0.0220
  Precision  : 0.4771 ± 0.0068


---
## Test set evaluation

In [ ]:
from src.pipeline import stage_load, stage_featurize
from src.evaluation.metrics import evaluate
from tabicl import TabICLClassifier

if TEST_CHECKPOINT.exists():
    print('Test result already exists — loading from checkpoint.')
    tm = json.loads(TEST_CHECKPOINT.read_text())
else:
    print('Running test set evaluation...')
    cfg = {**BASE_CONFIG, 'split_ratio': 0.8, 'subsample': 10000}

    data     = stage_load(cfg)
    features = stage_featurize(data, cfg)

    model = TabICLClassifier(**cfg['model_params'])
    model.fit(features.X_train, features.y_train)

    tm = evaluate(
        model,
        features.X_test,
        features.y_test,
        plot=True,
        title='TabICL improved + top-30 FS [TEST]',
        predict_batch_size=cfg.get('predict_batch_size'),
    )

    TEST_CHECKPOINT.write_text(json.dumps(
        {k: float(v) if isinstance(v, (int, float)) else v
         for k, v in tm.items()
         if not hasattr(v, '__len__') or isinstance(v, (str, list))},
        indent=2
    ))
    print('Saved.')

print(f"\nTest set:")
print(f"  ROC-AUC    : {tm['roc_auc']:.4f}")
print(f"  F1         : {tm['f1']:.4f}")
print(f"  Recall(pos): {tm['recall_pos']:.4f}")
print(f"  Threshold  : {tm['threshold']:.3f}")

---
## Comparison against all Task 3 models

In [ ]:
table_path = Path('../../outputs/tables/final_model_comparison.csv')
s = cv_result['summary']

task5_row = pd.DataFrame([{
    'model':            'tabicl_improved',
    'best_config':      f'tabicl (improved, top-{len(selected_features)} features)',
    'CV ROC-AUC':       f"{s['roc_auc_mean']:.4f} ± {s['roc_auc_std']:.4f}",
    'CV F1':            f"{s['f1_mean']:.4f} ± {s['f1_std']:.4f}",
    'CV Recall(pos)':   f"{s['recall_pos_mean']:.4f} ± {s['recall_pos_std']:.4f}",
    'Test ROC-AUC':     round(tm['roc_auc'], 4),
    'Test F1':          round(tm['f1'], 4),
    'Test Recall(pos)': round(tm['recall_pos'], 4),
    'Test Threshold':   round(tm['threshold'], 3),
}]).set_index('model')

if table_path.exists():
    existing = pd.read_csv(table_path, index_col='model')
    existing = existing[existing.index != 'tabicl_improved']
    combined = pd.concat([existing, task5_row])
else:
    combined = task5_row

combined.to_csv(table_path)
display(combined)

In [ ]:
numeric_cols = ['Test ROC-AUC', 'Test F1', 'Test Recall(pos)']
# Handle old column name if present
rename_map = {'Test Recall(<30)': 'Test Recall(pos)', 'CV Recall(<30)': 'CV Recall(pos)'}
plot_df = combined[numeric_cols].rename(columns=rename_map).dropna()

colors = ['#C44E52' if 'tabicl' in idx else '#4C72B0' for idx in plot_df.index]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    bars = ax.bar(plot_df.index, plot_df[col], color=colors, edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_title(col)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=25)
    sns.despine(ax=ax)

fig.suptitle('Test Set Performance — All Models (TabICL variants highlighted)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../../outputs/figures/task5_final_comparison.png', bbox_inches='tight', dpi=120)
plt.show()